<a href="https://colab.research.google.com/github/SANGHATI23/neurofhir-qc/blob/main/16A_NeuroFHIR_SAFE_Formal_Model_Checking_and_Mutation_Testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 16A — NeuroFHIR-SAFE
## Formal Model Checking + Safety Mutation Testing

**Main-study core. No human participants are required for this notebook.**

This notebook implements the formal layer of the NeuroFHIR-SAFE WISH design:

**Human-First evidence → explicit safety properties → finite-state workflow model → TLA+/TLC verification → counterexample traces → safety mutation testing**

It verifies the Evidence-First workflow, contrasts it with a controlled AI-First architecture, and intentionally breaks individual safeguards to show that the verification framework detects meaningful regressions.

### Formal properties

- **P1 Human-First ordering:** AI is never visible before an independent judgment.
- **P2 Uncertainty co-presentation:** AI is never visible without uncertainty/QC context.
- **P3 Low-QC finalization block.**
- **P4 Longitudinal-conflict acknowledgement before finalization.**
- **P5 Model identity required for final evidence.**
- **P6 Provenance required for final or rejected evidence.**
- **P7 Explicit human authorization required for finalization.**
- **P8 Rejection remains auditable.**
- **P9 Escalation remains non-final.**
- **P10 Task/result state consistency.**

The claims remain bounded: verification applies to the explicitly defined finite workflow model. It does **not** prove clinical safety or human performance.



Expected output directory:

`/content/drive/MyDrive/neurofhir-qc/wish_extension/neurofhir_safe/`

Key outputs:
- `wish_formal/NeuroFHIRSAFE.tla`
- Evidence-First / AI-First TLC configs
- M1–M9 mutation configs
- saved TLC logs and counterexamples
- `formal_results.json`
- `mutation_results.csv`
- `property_catalog.csv`
- reproducibility hashes

In [1]:
# Cell 1 — Mount Drive and configuration
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json, hashlib, subprocess, sys, re, os, textwrap, datetime, shutil
import pandas as pd

DRIVE_REPO_ROOT = Path("/content/drive/MyDrive/neurofhir-qc")
WISH_ROOT = DRIVE_REPO_ROOT / "wish_extension"
SAFE_ROOT = WISH_ROOT / "neurofhir_safe"
FORMAL_ROOT = SAFE_ROOT / "wish_formal"
RESULTS_ROOT = SAFE_ROOT / "artifacts" / "formal"
TLC_RESULTS = RESULTS_ROOT / "tlc_results"
COUNTEREXAMPLES = RESULTS_ROOT / "counterexamples"

for p in [SAFE_ROOT, FORMAL_ROOT, RESULTS_ROOT, TLC_RESULTS, COUNTEREXAMPLES]:
    p.mkdir(parents=True, exist_ok=True)

FROZEN_APP = WISH_ROOT / "final_wish_pilot" / "participant_app"
assert FROZEN_APP.exists(), f"Frozen participant app not found: {FROZEN_APP}"

# Pinned official TLA+ tools release. Change only if the release URL becomes unavailable.
TLA2TOOLS_URL = "https://github.com/tlaplus/tlaplus/releases/download/v1.8.0/tla2tools.jar"
TLA2TOOLS_JAR = FORMAL_ROOT / "tla2tools.jar"

print("SAFE root:", SAFE_ROOT)
print("Frozen app:", FROZEN_APP)

Mounted at /content/drive
SAFE root: /content/drive/MyDrive/neurofhir-qc/wish_extension/neurofhir_safe
Frozen app: /content/drive/MyDrive/neurofhir-qc/wish_extension/final_wish_pilot/participant_app


In [2]:
# Cell 2 — Freeze and hash the current application build

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def tree_manifest(root: Path):
    rows = []
    for p in sorted(root.rglob("*")):
        if p.is_file():
            rows.append({
                "path": p.relative_to(root).as_posix(),
                "sha256": sha256_file(p),
                "bytes": p.stat().st_size,
            })
    return rows

app_manifest = tree_manifest(FROZEN_APP)
assert app_manifest, "Frozen app is empty."

app_manifest_path = RESULTS_ROOT / "frozen_app_manifest.json"
app_manifest_path.write_text(json.dumps(app_manifest, indent=2), encoding="utf-8")

manifest_hash = hashlib.sha256(
    json.dumps(app_manifest, sort_keys=True).encode("utf-8")
).hexdigest()

print("Frozen files:", len(app_manifest))
print("Frozen build-manifest SHA-256:", manifest_hash)
print("✅ Build freeze recorded.")

Frozen files: 14
Frozen build-manifest SHA-256: f2a40b8e5b0b793ac9a46b7266742dc9c487f439f303269e0fb84cf8ad0671ad
✅ Build freeze recorded.


In [3]:
# Cell 3 — Property catalog

properties = [
    ("P1","Human-First ordering",
     "aiVisible => initialJudgmentCommitted",
     "AI advice must never appear before an independent judgment exists."),
    ("P2","Uncertainty co-presentation",
     "aiVisible => uncertaintyVisible",
     "AI advice cannot appear without its QC/uncertainty context."),
    ("P3","Low-QC finalization block",
     "qcState in {LOW,FAILED} => resultStatus != FINAL",
     "Low/failed-QC evidence cannot silently become final."),
    ("P4","Conflict handling",
     "longitudinalConflict and FINAL => conflictAcknowledged",
     "Known longitudinal conflict must be acknowledged before finalization."),
    ("P5","Model identity",
     "FINAL => modelIdentityPresent",
     "Final AI-derived evidence must remain bound to model/version identity."),
    ("P6","Provenance completeness",
     "FINAL or ENTERED_IN_ERROR => provenanceComplete",
     "Accepted or rejected evidence must preserve lineage."),
    ("P7","Human authorization",
     "FINAL => reviewAction in {ACCEPT,AMEND}",
     "AI cannot finalize evidence without explicit human authorization."),
    ("P8","Rejection preservation",
     "REJECT => ENTERED_IN_ERROR and provenanceComplete",
     "Rejected evidence remains reconstructable."),
    ("P9","Escalation safety",
     "ESCALATE => resultStatus != FINAL",
     "Escalated cases remain non-final."),
    ("P10","Task-state consistency",
     "Result/review state agrees with Task state",
     "FHIR-style review workflow state must agree with evidence disposition."),
]

property_df = pd.DataFrame(
    properties,
    columns=["property_id","name","formal_condition","plain_language_requirement"]
)
property_path = RESULTS_ROOT / "property_catalog.csv"
property_df.to_csv(property_path, index=False)

display(property_df)
print("✅ Property catalog frozen:", property_path)

,property_id,name,formal_condition,plain_language_requirement
0,P1,Human-First ordering,aiVisible => initialJudgmentCommitted,AI advice must never appear before an independ...
1,P2,Uncertainty co-presentation,aiVisible => uncertaintyVisible,AI advice cannot appear without its QC/uncerta...
2,P3,Low-QC finalization block,"qcState in {LOW,FAILED} => resultStatus != FINAL",Low/failed-QC evidence cannot silently become ...
3,P4,Conflict handling,longitudinalConflict and FINAL => conflictAckn...,Known longitudinal conflict must be acknowledg...
4,P5,Model identity,FINAL => modelIdentityPresent,Final AI-derived evidence must remain bound to...
5,P6,Provenance completeness,FINAL or ENTERED_IN_ERROR => provenanceComplete,Accepted or rejected evidence must preserve li...
6,P7,Human authorization,"FINAL => reviewAction in {ACCEPT,AMEND}",AI cannot finalize evidence without explicit h...
7,P8,Rejection preservation,REJECT => ENTERED_IN_ERROR and provenanceComplete,Rejected evidence remains reconstructable.
8,P9,Escalation safety,ESCALATE => resultStatus != FINAL,Escalated cases remain non-final.
9,P10,Task-state consistency,Result/review state agrees with Task state,FHIR-style review workflow state must agree wi...


✅ Property catalog frozen: /content/drive/MyDrive/neurofhir-qc/wish_extension/neurofhir_safe/artifacts/formal/property_catalog.csv


In [4]:
# Cell 4 — Write the finite-state TLA+ specification

TLA_SPEC = r"""
---- MODULE NeuroFHIRSAFE ----
EXTENDS Naturals, TLC

CONSTANTS
  AI_FIRST,
  MUT_EARLY_AI,
  MUT_HIDE_UNCERTAINTY,
  MUT_ALLOW_LOW_QC_FINAL,
  MUT_ALLOW_UNACK_CONFLICT_FINAL,
  MUT_ALLOW_NO_MODEL_FINAL,
  MUT_NO_PROV_REJECT,
  MUT_AI_AUTOFINAL,
  MUT_FINAL_ESCALATE,
  MUT_TASK_MISMATCH

VARIABLES
  phase,
  initialJudgmentCommitted,
  aiVisible,
  uncertaintyVisible,
  qcState,
  longitudinalConflict,
  conflictAcknowledged,
  provenanceComplete,
  modelIdentityPresent,
  reviewAction,
  resultStatus,
  taskStatus

vars ==
  << phase,
     initialJudgmentCommitted,
     aiVisible,
     uncertaintyVisible,
     qcState,
     longitudinalConflict,
     conflictAcknowledged,
     provenanceComplete,
     modelIdentityPresent,
     reviewAction,
     resultStatus,
     taskStatus >>

Phases ==
  {"OPEN","EVIDENCE","INITIAL_JUDGMENT","AI_REVEAL",
   "RECONCILE","FINAL","REJECTED","ESCALATED"}

QCStates == {"HIGH","LOW","FAILED"}
ReviewActions == {"NONE","ACCEPT","AMEND","REJECT","ESCALATE"}
ResultStatuses == {"PRELIMINARY","FINAL","ENTERED_IN_ERROR"}
TaskStatuses == {"REQUESTED","ON_HOLD","COMPLETED","REJECTED"}

Init ==
  /\ phase = "OPEN"
  /\ initialJudgmentCommitted = FALSE
  /\ aiVisible = FALSE
  /\ uncertaintyVisible = FALSE
  /\ qcState \in QCStates
  /\ longitudinalConflict \in BOOLEAN
  /\ conflictAcknowledged = FALSE
  /\ provenanceComplete \in BOOLEAN
  /\ modelIdentityPresent \in BOOLEAN
  /\ reviewAction = "NONE"
  /\ resultStatus = "PRELIMINARY"
  /\ taskStatus = "REQUESTED"

OpenEvidence ==
  /\ phase = "OPEN"
  /\ phase' = "EVIDENCE"
  /\ IF AI_FIRST \/ MUT_EARLY_AI
        THEN /\ aiVisible' = TRUE
             /\ uncertaintyVisible' = ~MUT_HIDE_UNCERTAINTY
        ELSE /\ aiVisible' = aiVisible
             /\ uncertaintyVisible' = uncertaintyVisible
  /\ UNCHANGED
       << initialJudgmentCommitted, qcState, longitudinalConflict,
          conflictAcknowledged, provenanceComplete, modelIdentityPresent,
          reviewAction, resultStatus, taskStatus >>

CommitInitialJudgment ==
  /\ phase = "EVIDENCE"
  /\ phase' = "INITIAL_JUDGMENT"
  /\ initialJudgmentCommitted' = TRUE
  /\ UNCHANGED
       << aiVisible, uncertaintyVisible, qcState, longitudinalConflict,
          conflictAcknowledged, provenanceComplete, modelIdentityPresent,
          reviewAction, resultStatus, taskStatus >>

RevealAI ==
  /\ phase = "INITIAL_JUDGMENT"
  /\ phase' = "AI_REVEAL"
  /\ aiVisible' = TRUE
  /\ uncertaintyVisible' = ~MUT_HIDE_UNCERTAINTY
  /\ UNCHANGED
       << initialJudgmentCommitted, qcState, longitudinalConflict,
          conflictAcknowledged, provenanceComplete, modelIdentityPresent,
          reviewAction, resultStatus, taskStatus >>

EnterReconcile ==
  /\ phase = "AI_REVEAL"
  /\ phase' = "RECONCILE"
  /\ UNCHANGED
       << initialJudgmentCommitted, aiVisible, uncertaintyVisible, qcState,
          longitudinalConflict, conflictAcknowledged, provenanceComplete,
          modelIdentityPresent, reviewAction, resultStatus, taskStatus >>

AcknowledgeConflict ==
  /\ phase = "RECONCILE"
  /\ longitudinalConflict = TRUE
  /\ conflictAcknowledged = FALSE
  /\ conflictAcknowledged' = TRUE
  /\ UNCHANGED
       << phase, initialJudgmentCommitted, aiVisible, uncertaintyVisible,
          qcState, longitudinalConflict, provenanceComplete,
          modelIdentityPresent, reviewAction, resultStatus, taskStatus >>

AcceptFinal ==
  /\ phase = "RECONCILE"
  /\ (qcState = "HIGH" \/ MUT_ALLOW_LOW_QC_FINAL)
  /\ (~longitudinalConflict \/ conflictAcknowledged \/
       MUT_ALLOW_UNACK_CONFLICT_FINAL)
  /\ (modelIdentityPresent \/ MUT_ALLOW_NO_MODEL_FINAL)
  /\ provenanceComplete
  /\ phase' = "FINAL"
  /\ reviewAction' = "ACCEPT"
  /\ resultStatus' = "FINAL"
  /\ taskStatus' = IF MUT_TASK_MISMATCH THEN "REQUESTED" ELSE "COMPLETED"
  /\ UNCHANGED
       << initialJudgmentCommitted, aiVisible, uncertaintyVisible, qcState,
          longitudinalConflict, conflictAcknowledged, provenanceComplete,
          modelIdentityPresent >>

AmendFinal ==
  /\ phase = "RECONCILE"
  /\ (qcState = "HIGH" \/ MUT_ALLOW_LOW_QC_FINAL)
  /\ (~longitudinalConflict \/ conflictAcknowledged \/
       MUT_ALLOW_UNACK_CONFLICT_FINAL)
  /\ (modelIdentityPresent \/ MUT_ALLOW_NO_MODEL_FINAL)
  /\ provenanceComplete
  /\ phase' = "FINAL"
  /\ reviewAction' = "AMEND"
  /\ resultStatus' = "FINAL"
  /\ taskStatus' = IF MUT_TASK_MISMATCH THEN "REQUESTED" ELSE "COMPLETED"
  /\ UNCHANGED
       << initialJudgmentCommitted, aiVisible, uncertaintyVisible, qcState,
          longitudinalConflict, conflictAcknowledged, provenanceComplete,
          modelIdentityPresent >>

RejectEvidence ==
  /\ phase = "RECONCILE"
  /\ (provenanceComplete \/ MUT_NO_PROV_REJECT)
  /\ phase' = "REJECTED"
  /\ reviewAction' = "REJECT"
  /\ resultStatus' = "ENTERED_IN_ERROR"
  /\ provenanceComplete' =
       IF MUT_NO_PROV_REJECT THEN FALSE ELSE provenanceComplete
  /\ taskStatus' = IF MUT_TASK_MISMATCH THEN "COMPLETED" ELSE "REJECTED"
  /\ UNCHANGED
       << initialJudgmentCommitted, aiVisible, uncertaintyVisible, qcState,
          longitudinalConflict, conflictAcknowledged, modelIdentityPresent >>

Escalate ==
  /\ phase = "RECONCILE"
  /\ phase' = "ESCALATED"
  /\ reviewAction' = "ESCALATE"
  /\ resultStatus' =
       IF MUT_FINAL_ESCALATE THEN "FINAL" ELSE "PRELIMINARY"
  /\ taskStatus' = "ON_HOLD"
  /\ UNCHANGED
       << initialJudgmentCommitted, aiVisible, uncertaintyVisible, qcState,
          longitudinalConflict, conflictAcknowledged, provenanceComplete,
          modelIdentityPresent >>

AutoFinalize ==
  /\ MUT_AI_AUTOFINAL
  /\ aiVisible = TRUE
  /\ phase \in {"AI_REVEAL","RECONCILE"}
  /\ phase' = "FINAL"
  /\ reviewAction' = "NONE"
  /\ resultStatus' = "FINAL"
  /\ taskStatus' = "COMPLETED"
  /\ UNCHANGED
       << initialJudgmentCommitted, aiVisible, uncertaintyVisible, qcState,
          longitudinalConflict, conflictAcknowledged, provenanceComplete,
          modelIdentityPresent >>

Next ==
  \/ OpenEvidence
  \/ CommitInitialJudgment
  \/ RevealAI
  \/ EnterReconcile
  \/ AcknowledgeConflict
  \/ AcceptFinal
  \/ AmendFinal
  \/ RejectEvidence
  \/ Escalate
  \/ AutoFinalize

Spec == Init /\ [][Next]_vars

TypeOK ==
  /\ phase \in Phases
  /\ initialJudgmentCommitted \in BOOLEAN
  /\ aiVisible \in BOOLEAN
  /\ uncertaintyVisible \in BOOLEAN
  /\ qcState \in QCStates
  /\ longitudinalConflict \in BOOLEAN
  /\ conflictAcknowledged \in BOOLEAN
  /\ provenanceComplete \in BOOLEAN
  /\ modelIdentityPresent \in BOOLEAN
  /\ reviewAction \in ReviewActions
  /\ resultStatus \in ResultStatuses
  /\ taskStatus \in TaskStatuses

P1 == aiVisible => initialJudgmentCommitted
P2 == aiVisible => uncertaintyVisible
P3 == (qcState \in {"LOW","FAILED"}) => resultStatus # "FINAL"
P4 == (longitudinalConflict /\ resultStatus = "FINAL") =>
        conflictAcknowledged
P5 == resultStatus = "FINAL" => modelIdentityPresent
P6 == resultStatus \in {"FINAL","ENTERED_IN_ERROR"} =>
        provenanceComplete
P7 == resultStatus = "FINAL" =>
        reviewAction \in {"ACCEPT","AMEND"}
P8 == reviewAction = "REJECT" =>
        (resultStatus = "ENTERED_IN_ERROR" /\ provenanceComplete)
P9 == reviewAction = "ESCALATE" => resultStatus # "FINAL"
P10 ==
  /\ (resultStatus = "FINAL" => taskStatus = "COMPLETED")
  /\ (reviewAction = "REJECT" => taskStatus = "REJECTED")
  /\ (reviewAction = "ESCALATE" => taskStatus # "COMPLETED")

====
"""

spec_path = FORMAL_ROOT / "NeuroFHIRSAFE.tla"
spec_path.write_text(TLA_SPEC.strip() + "\n", encoding="utf-8")

print(spec_path)
print("TLA+ SHA-256:", sha256_file(spec_path))

/content/drive/MyDrive/neurofhir-qc/wish_extension/neurofhir_safe/wish_formal/NeuroFHIRSAFE.tla
TLA+ SHA-256: 6fb33631b365f4965ac8c9acd964f6f2be0408495ae83c003c9d2a61cfdae3de


In [5]:
# Cell 5 — Write Evidence-First, AI-First, and M1–M9 configurations

CONST_NAMES = [
    "AI_FIRST",
    "MUT_EARLY_AI",
    "MUT_HIDE_UNCERTAINTY",
    "MUT_ALLOW_LOW_QC_FINAL",
    "MUT_ALLOW_UNACK_CONFLICT_FINAL",
    "MUT_ALLOW_NO_MODEL_FINAL",
    "MUT_NO_PROV_REJECT",
    "MUT_AI_AUTOFINAL",
    "MUT_FINAL_ESCALATE",
    "MUT_TASK_MISMATCH",
]

def cfg_text(constants, invariants):
    lines = ["SPECIFICATION Spec", "CHECK_DEADLOCK FALSE", "CONSTANTS"]
    for k in CONST_NAMES:
        lines.append(f"  {k} = {'TRUE' if constants.get(k, False) else 'FALSE'}")
    lines.append("INVARIANTS")
    lines.append("  TypeOK")
    for inv in invariants:
        lines.append(f"  {inv}")
    return "\n".join(lines) + "\n"

base = {k: False for k in CONST_NAMES}

configs = {}

configs["EvidenceFirst.cfg"] = (base.copy(), [f"P{i}" for i in range(1,11)], "PASS")
ai = base.copy()
ai["AI_FIRST"] = True
configs["AIFirst.cfg"] = (ai, ["P1"], "P1")

mutants = {
    "M1_early_ai.cfg": ("MUT_EARLY_AI", "P1"),
    "M2_hide_uncertainty.cfg": ("MUT_HIDE_UNCERTAINTY", "P2"),
    "M3_low_qc_final.cfg": ("MUT_ALLOW_LOW_QC_FINAL", "P3"),
    "M4_unack_conflict.cfg": ("MUT_ALLOW_UNACK_CONFLICT_FINAL", "P4"),
    "M5_no_model_identity.cfg": ("MUT_ALLOW_NO_MODEL_FINAL", "P5"),
    "M6_no_provenance_reject.cfg": ("MUT_NO_PROV_REJECT", "P6"),
    "M7_ai_autofinal.cfg": ("MUT_AI_AUTOFINAL", "P7"),
    "M8_final_escalation.cfg": ("MUT_FINAL_ESCALATE", "P9"),
    "M9_task_mismatch.cfg": ("MUT_TASK_MISMATCH", "P10"),
}

for name, (flag, expected) in mutants.items():
    c = base.copy()
    c[flag] = True
    configs[name] = (c, [expected], expected)

for name, (constants, invariants, expected) in configs.items():
    (FORMAL_ROOT / name).write_text(
        cfg_text(constants, invariants),
        encoding="utf-8"
    )

print("Written configs:")
for name in configs:
    print(" -", name)

Written configs:
 - EvidenceFirst.cfg
 - AIFirst.cfg
 - M1_early_ai.cfg
 - M2_hide_uncertainty.cfg
 - M3_low_qc_final.cfg
 - M4_unack_conflict.cfg
 - M5_no_model_identity.cfg
 - M6_no_provenance_reject.cfg
 - M7_ai_autofinal.cfg
 - M8_final_escalation.cfg
 - M9_task_mismatch.cfg


In [6]:
# Cell 6 — Install / verify Java and TLA+ tools

java_check = subprocess.run(
    ["java","-version"], text=True, capture_output=True
)
assert java_check.returncode == 0, (
    "Java is required for TLC. Colab normally provides it."
)
print((java_check.stderr or java_check.stdout).splitlines()[0])

if not TLA2TOOLS_JAR.exists():
    import requests
    print("Downloading official TLA+ tools...")
    r = requests.get(TLA2TOOLS_URL, timeout=120)
    r.raise_for_status()
    TLA2TOOLS_JAR.write_bytes(r.content)

assert TLA2TOOLS_JAR.stat().st_size > 1_000_000, (
    "Downloaded tla2tools.jar looks incomplete."
)

print("TLA+ tools:", TLA2TOOLS_JAR)
print("SHA-256:", sha256_file(TLA2TOOLS_JAR))

openjdk version "21.0.12" 2026-07-21
TLA+ tools: /content/drive/MyDrive/neurofhir-qc/wish_extension/neurofhir_safe/wish_formal/tla2tools.jar
SHA-256: eab20b266c1dd9cedec13a343c471be75dcced18255d9dd19cbf1a249fb8de31


In [7]:
# Cell 7 — TLC runner and parser

def run_tlc(cfg_name, timeout=180):
    cfg_path = FORMAL_ROOT / cfg_name
    cmd = [
        "java", "-cp", str(TLA2TOOLS_JAR),
        "tlc2.TLC",
        "-workers", "auto",
        "-config", str(cfg_path),
        str(spec_path),
    ]
    proc = subprocess.run(
        cmd,
        cwd=FORMAL_ROOT,
        text=True,
        capture_output=True,
        timeout=timeout,
    )
    output = (proc.stdout or "") + "\n" + (proc.stderr or "")
    log_path = TLC_RESULTS / f"{Path(cfg_name).stem}.log"
    log_path.write_text(output, encoding="utf-8")

    states = None
    distinct = None
    m = re.search(r"(\d+)\s+states generated", output, re.I)
    if m:
        states = int(m.group(1))
    m = re.search(r"(\d+)\s+distinct states found", output, re.I)
    if m:
        distinct = int(m.group(1))

    violated = None
    m = re.search(r"Invariant\s+(P\d+)\s+is violated", output, re.I)
    if m:
        violated = m.group(1).upper()

    return {
        "config": cfg_name,
        "returncode": proc.returncode,
        "states_generated": states,
        "distinct_states": distinct,
        "violated_property": violated,
        "log": str(log_path),
        "output": output,
    }

print("✅ TLC runner ready.")

✅ TLC runner ready.


In [8]:
# Cell 8 — Verify the Evidence-First model

evidence_result = run_tlc("EvidenceFirst.cfg")

print("Return code:", evidence_result["returncode"])
print("States generated:", evidence_result["states_generated"])
print("Distinct states:", evidence_result["distinct_states"])
print("Violation:", evidence_result["violated_property"])

assert evidence_result["violated_property"] is None, (
    "Evidence-First violated an invariant. Inspect: "
    + evidence_result["log"]
)
assert evidence_result["returncode"] == 0, (
    "TLC did not complete successfully. Inspect: "
    + evidence_result["log"]
)

print("✅ Evidence-First P1–P10 verification: PASS")

Return code: 0
States generated: 190
Distinct states: 190
Violation: None
✅ Evidence-First P1–P10 verification: PASS


In [9]:
# Cell 9 — Controlled AI-First architectural counterexample

ai_result = run_tlc("AIFirst.cfg")

print("States generated:", ai_result["states_generated"])
print("Distinct states:", ai_result["distinct_states"])
print("Violated property:", ai_result["violated_property"])

assert ai_result["violated_property"] == "P1", (
    "AI-First comparator did not produce the expected P1 counterexample. "
    "Inspect: " + ai_result["log"]
)

# Preserve the complete TLC trace as the counterexample artifact.
counterexample_path = COUNTEREXAMPLES / "AIFirst_P1_counterexample.txt"
counterexample_path.write_text(ai_result["output"], encoding="utf-8")

print("✅ AI-First admits the expected P1 Human-First-ordering counterexample.")
print("Saved:", counterexample_path)

States generated: 26
Distinct states: 26
Violated property: P1
✅ AI-First admits the expected P1 Human-First-ordering counterexample.
Saved: /content/drive/MyDrive/neurofhir-qc/wish_extension/neurofhir_safe/artifacts/formal/counterexamples/AIFirst_P1_counterexample.txt


In [10]:
# Cell 10 — Run M1–M9 safety mutation testing

mutation_rows = []

for cfg_name, (_, _, expected_property) in configs.items():
    if not cfg_name.startswith("M"):
        continue

    result = run_tlc(cfg_name)
    detected = result["violated_property"] == expected_property

    mutation_rows.append({
        "mutant": Path(cfg_name).stem,
        "expected_property": expected_property,
        "observed_violation": result["violated_property"],
        "detected": detected,
        "states_generated": result["states_generated"],
        "distinct_states": result["distinct_states"],
        "log": result["log"],
    })

    if result["violated_property"]:
        (COUNTEREXAMPLES / f"{Path(cfg_name).stem}_{result['violated_property']}.txt").write_text(
            result["output"], encoding="utf-8"
        )

mutation_df = pd.DataFrame(mutation_rows)
display(mutation_df)

mutation_path = RESULTS_ROOT / "mutation_results.csv"
mutation_df.to_csv(mutation_path, index=False)

detection_rate = float(mutation_df["detected"].mean()) if len(mutation_df) else 0.0

print(f"Mutation detection rate: {mutation_df['detected'].sum()}/{len(mutation_df)} = {detection_rate:.3f}")

assert mutation_df["detected"].all(), (
    "At least one deliberate safety regression was not detected. "
    "Do not proceed to submission claims until fixed."
)

print("✅ M1–M9 mutation sensitivity: PASS")

,mutant,expected_property,observed_violation,detected,states_generated,distinct_states,log
0,M1_early_ai,P1,P1,True,26,26,/content/drive/MyDrive/neurofhir-qc/wish_exten...
1,M2_hide_uncertainty,P2,P2,True,74,74,/content/drive/MyDrive/neurofhir-qc/wish_exten...
2,M3_low_qc_final,P3,P3,True,157,155,/content/drive/MyDrive/neurofhir-qc/wish_exten...
3,M4_unack_conflict,P4,P4,True,193,189,/content/drive/MyDrive/neurofhir-qc/wish_exten...
4,M5_no_model_identity,P5,P5,True,171,169,/content/drive/MyDrive/neurofhir-qc/wish_exten...
5,M6_no_provenance_reject,P6,P6,True,122,122,/content/drive/MyDrive/neurofhir-qc/wish_exten...
6,M7_ai_autofinal,P7,P7,True,98,98,/content/drive/MyDrive/neurofhir-qc/wish_exten...
7,M8_final_escalation,P9,P9,True,121,121,/content/drive/MyDrive/neurofhir-qc/wish_exten...
8,M9_task_mismatch,P10,P10,True,123,123,/content/drive/MyDrive/neurofhir-qc/wish_exten...


Mutation detection rate: 9/9 = 1.000
✅ M1–M9 mutation sensitivity: PASS


In [11]:
# Cell 11 — Formal-results package

formal_results = {
    "generated_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "study": "NeuroFHIR-SAFE",
    "claim_boundary": (
        "Finite workflow-model verification only; no claim of clinical safety "
        "or human-performance improvement."
    ),
    "frozen_app_manifest_sha256": manifest_hash,
    "tla_spec_sha256": sha256_file(spec_path),
    "tla2tools_sha256": sha256_file(TLA2TOOLS_JAR),
    "evidence_first": {
        k: v for k, v in evidence_result.items()
        if k != "output"
    },
    "ai_first": {
        k: v for k, v in ai_result.items()
        if k != "output"
    },
    "mutation_detection": {
        "detected": int(mutation_df["detected"].sum()),
        "total": int(len(mutation_df)),
        "rate": detection_rate,
    },
    "properties": property_df.to_dict(orient="records"),
}

formal_results_path = RESULTS_ROOT / "formal_results.json"
formal_results_path.write_text(
    json.dumps(formal_results, indent=2),
    encoding="utf-8"
)

print("✅ Formal results:", formal_results_path)

✅ Formal results: /content/drive/MyDrive/neurofhir-qc/wish_extension/neurofhir_safe/artifacts/formal/formal_results.json


In [12]:
# Cell 12 — Final formal-verification gate

gate = {
    "Evidence-First P1–P10 verified": evidence_result["returncode"] == 0 and evidence_result["violated_property"] is None,
    "AI-First P1 counterexample": ai_result["violated_property"] == "P1",
    "M1–M9 all detected": bool(mutation_df["detected"].all()),
    "Frozen build manifest saved": app_manifest_path.exists(),
    "Formal artifacts saved": formal_results_path.exists(),
}

for k, v in gate.items():
    print(f"{'PASS' if v else 'FAIL':4}  {k}")

assert all(gate.values())

print("=" * 82)
print("✅ NOTEBOOK 16A NEUROFHIR-SAFE FORMAL CORE GATE: TRUE")
print("=" * 82)
print("NEXT → Notebook 16B: runtime trace + FHIR conformance.")

PASS  Evidence-First P1–P10 verified
PASS  AI-First P1 counterexample
PASS  M1–M9 all detected
PASS  Frozen build manifest saved
PASS  Formal artifacts saved
✅ NOTEBOOK 16A NEUROFHIR-SAFE FORMAL CORE GATE: TRUE
NEXT → Notebook 16B: runtime trace + FHIR conformance.
